# 5 Advanced Visualizations

### 📊 What is the visualization about?
We want to create a shot chart for a given NHL team, showing the location of its shots on the ice to visualize shooting patterns and identify areas of offensive strength.

In [60]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from ift6758.features.simple_visualizations import games_list
import plotly.graph_objects as go

- `number_games()` calculates the number of games played by each team during a given season.

In [61]:
def number_games(season) -> dict:
    # Get all the games dataframe
    df_games = games_list(season, events=False)

    # Keep regular season and playoffs only (game type code in the game id)
    df_games = df_games[df_games["game_type"].isin([2, 3])]

    # Calculate the number of game for each team
    number_games_per_team = {}
    for game in df_games.itertuples():
        homeTeam_id = getattr(game, "homeTeam_name")
        awayTeam_id = getattr(game, "awayTeam_name")
        number_games_per_team[homeTeam_id] = number_games_per_team.get(homeTeam_id, 0) + 1
        number_games_per_team[awayTeam_id] = number_games_per_team.get(awayTeam_id, 0) + 1

    return number_games_per_team

- `shot_data` contains all the information needed to generate the desired visualization. It returns a DataFrame containing, for each team, its name, the number of games played during the season, and the coordinates of every shot and goal recorded throughout the season.

In [136]:

def shot_data(season):
    """
    Gets the data from json (possibly by calling the distance function) for the season and extracts the shot coordinates to a more usable format
    :param season: 4 digit year of the season start
    :return: a dataframe containing each team's name, number of games, and an array of x,y shot coordinates (1 line per shot, columns x and y)
    """
    # Get each team's number of games
    team_game_number = number_games(season)

    # Link each team_id to its abbrev
    df_games = games_list(season, events=False)
    id_to_abbrev = {}
    for game in df_games.itertuples():
        id_to_abbrev[getattr(game, "homeTeam_id")] = getattr(game, "homeTeam_name")
        id_to_abbrev[getattr(game, "awayTeam_id")] = getattr(game, "awayTeam_name")

    # Get the shots and goals dataframe for the given season
    df_events = games_list(season)

    # Keep regular season and playoffs
    df_shots = df_events[
        df_events["game_type"].isin(["regular season", "playoffs"])
    ].copy()

    # Put all the shots on the same side (use same logic as distance() from simple_visualization.py)
    home_shoots = df_shots["team_id"] == df_shots["homeTeam_id"]
    attacks_left = np.where(
        home_shoots,
        df_shots["homeTeamDefendingSide"] == "right",
        df_shots["homeTeamDefendingSide"] == "left"
    )

    # Flip the shots by 180°­­­­ if they were taken from the left side of the pitch
    df_shots.loc[attacks_left, ["x_coord", "y_coord"]] *= -1

    rows = []
    for team_id, df_team in df_shots.groupby("team_id"):
        team = id_to_abbrev[team_id]

        coords = df_team[["x_coord", "y_coord"]]
        if not np.isfinite(coords).all().all():
            coords = coords[np.isfinite(coords)]

        rows.append({
            "team": team,
            "games": team_game_number[team],
            "shots": coords.T
        })

    return pd.DataFrame(rows).T

#print(shot_data(2022).head())

In [ ]:

class ShotMap:

    def __init__(self, season):
        """

        :param season: 4 digit year of the season start
        """
        self.season = season
        self.shots = shot_data(season).copy()
        self.kdes = []
        self.grid = [] # list of shot densities for each team

    def calculate_kdes1(self):
        """
        This calculates league average shot rate per hour (league_sph) and excess shot rate per hour for each team, in the form of shot density using kde. Also saves raw difference in shots per team hour
        :return:
        """
        # first, calculate kde for league average and save the factor, which will be given to the team kdes
        coords_league = np.ndarray(self.shots["shots"])
        print("LEAGUE COORDS: " + coords_league)
        #for team_coords in self.shots["shots"]:
        #    coords_league.append((team_coords["x_coord"], team_coords["y_coord"]))

        for team_coords in self.shots["shots"]:
            print(team_coords)
            coords_league.append(team_coords)

        self.league_kde = self.kdes.append(stats.gaussian_kde(pd.concat(coords_league)))
        factor = self.league_kde.covariance_factor()

        # calculate shot rate per hour density for each team using kde:
        for team_coords in self.shots["shots"]:
            self.kdes.append(stats.gaussian_kde(team_coords, bw_method=factor))

        # densities will need to be weighed by shot rate

        # convert teams' sph to excess sph using league average
        # calculate raw difference in shots per team hour
        self.shot_rate = []
        for team_coords in self.shots["shots"]:
            self.shot_rate.append(team_coords.shape[1])

    def calculate_kdes(self):
        """
        This calculates league average shot rate per hour (league_sph) and excess shot rate per hour for each team, in the form of shot density using kde. Also saves raw difference in shots per team hour
        :return:
        """
        # first, calculate kde for league average and save the factor, which will be given to the team kdes
        
        # GEMINI LOGIC: used to remove inf or NaN values in the shots coordinates
        coords = pd.concat(self.shots.iloc[2,:].values, axis=1).replace([np.inf, -np.inf], np.nan)
        coords = coords.dropna(axis=1)
        coords = coords.dropna(axis=0)
        
        self.league_kde = stats.gaussian_kde(coords)
        factor = self.league_kde.covariance_factor()

        # calculate shot rate per hour density for each team using kde:
        for index, team in self.shots.items():
            # GEMINI LOGIC: used to remove inf or NaN values in the shots coordinates
            team_shots = team["shots"].values
            team_shots = team_shots[np.isfinite(team_shots)]

            self.kdes.append(stats.gaussian_kde(team_shots, bw_method=factor))

        # densities will need to be weighed by shot rate

        # convert teams' sph to excess sph using league average
        # calculate total shots per hour for each team :
        self.shot_rate = np.zeros([self.shots.shape[1],1], dtype=float)
        self.league_sn = 0
        self.league_hrs = 0
        for ind, team in self.shots.items():
            self.shot_rate[ind] = (team["shots"].values.shape[1] / (2*team["games"]) )
            self.league_sn += team["shots"].values.shape[1]
            self.league_hrs += (2*team["games"])
        self.league_avg = self.league_sn/self.league_hrs


    def create_grid(self):

        # Create the grid in the shape of the rink:
        X, Y = np.mgrid[-100:100:200j, -42.5:42.5:85j]
        positions = np.vstack([X.ravel(), Y.ravel()])

        league_avg = self.league_avg*(np.reshape(self.league_kde(positions).T, X.shape))
        # fill the grid list with difference team-league:
        for ind, team in self.shots.items():
            self.grid.append(100*(self.shot_rate[ind]*np.reshape(self.kdes[ind](positions).T, X.shape) - league_avg) )
        return self.grid





In [64]:
# Function from  demos, but with added goal lines
def draw_rink(fig):
    # A 200-foot by 85-foot rectangle, with center line, o-zone lines and goal lines.
    fig.add_shape(type="rect", x0=-100, x1=100, y0=-42.5, y1=42.5,
                  line_color="#7f95aa")
    for x, colour in [(-89, "#9bc2e6"),(-25, "#9bc2e6"), (0, "#e6a0a0"), (25, "#9bc2e6"), (89, "#9bc2e6")]:
        fig.add_vline(x=x, line_color=colour)
    fig.update_xaxes(range=[-103, 103], title="x (ft)", constrain="domain")
    fig.update_yaxes(range=[-46, 46], title="y (ft)", scaleanchor="x", scaleratio=1)
    fig.update_layout(template="plotly_white", height=460,
                      margin=dict(l=55, r=25, t=55, b=50))
    return fig

In [153]:
def interactive_shot_map(shot_map:(ShotMap)):
    """
    :param shot_map: ShotMap object
    :return: Creates html interactive figure ?
    """
    # checks if shot_map has kdes, if not, calls shot_map.calculate_kde()
    if len(shot_map.kdes) == 0:
        shot_map.calculate_kdes()
    df = shot_map.shots

    # get the grid of shot densities :
    if len(shot_map.grid) != 0:
        grid = shot_map.grid
    else:
        grid = shot_map.create_grid()

    team_list = df.iloc[0,:]

    # The dict of what changes depending on the team :
    teams = []
    for i in range(len(team_list)):
        it = dict(label = team_list[i],
                  args = [{
                      "z": grid[i].T,
                      "name" : f"Shots per team-hour above league average: {(shot_map.shot_rate[i]-shot_map.league_avg)[0]:.04} shots/team-hour"}],
                  method="restyle")
        teams.append(it)

    # CODE TAKEN FROM PLOTLY DEMO
    # create figure
    fig = go.Figure()

    fig = draw_rink(fig)

    # Add surface trace
    # and show the team's shots per team-hour above average
    # The heatmap is initialized to the first team
    fig.add_trace(go.Heatmap(x = np.linspace(-100,100,200),
                             y = np.linspace(-43,43,85),
                             z = [grid[0].T],
                             colorscale="RdBu",
                             zmin = -0.25,
                             zmax = 0.25,
                             name = f"Shots per team-hour above league average: {(shot_map.shot_rate[0]-shot_map.league_avg)[0]:.04} shots/team-hour", showlegend=True ))

    # Update plot sizing
    fig.update_layout(
        #width=800,
        #height=900,
        #autosize=False,
        showlegend=True,
        margin=dict(t=0, b=0, l=0, r=0),
        template="plotly_white",
        legend=dict(
            xanchor="left",
            yanchor="top",
            x=0,
            y=1)
    )

    # Update 3D scene options
    fig.update_scenes(
        aspectratio=dict(x=1, y=1, z=0.7),
        aspectmode="manual"
    )

    # Add dropdown
    fig.update_layout(
        updatemenus=[
            dict(
                buttons = teams, #Changed code here
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
        ]
    )

    # Add annotation
    fig.update_layout(
        annotations=[
            dict(text="Team:", showarrow=False,
            x=0, y=1.085, yref="paper", align="left")
        ]
    )


    # From demo in blog template
    fig.write_html("../../figures/interactive_figure_" + str(shot_map.season) + ".html", include_plotlyjs="cdn", full_html=False)
    fig.show()

In [164]:
shots2016 = ShotMap(2016)
# interactive_shot_map(shots2016)

KeyboardInterrupt: 

In [166]:
il = pd.concat(shots2016.shots.iloc[2,:].values, axis=1)
il.dropna(axis=1)
il.dropna(axis=0)
print(il)
print(pd.concat(shots2016.shots.iloc[2,:].values, axis=1))

         579    584    585    588    590    594    596    597    598    599    \
x_coord   42.0   40.0   84.0   52.0   54.0   71.0   36.0   76.0   35.0   60.0   
y_coord   24.0  -18.0   -8.0    9.0  -18.0   10.0  -32.0   14.0  -35.0    8.0   

         ...  74164  74165  74166  74173  74175  74176  74177  74178  74190  \
x_coord  ...   80.0   36.0   73.0   81.0   60.0   51.0   46.0   34.0   53.0   
y_coord  ...    3.0  -29.0  -38.0    4.0  -35.0  -29.0   28.0  -38.0  -34.0   

         74192  
x_coord   46.0  
y_coord  -32.0  

[2 rows x 80399 columns]
         579    584    585    588    590    594    596    597    598    599    \
x_coord   42.0   40.0   84.0   52.0   54.0   71.0   36.0   76.0   35.0   60.0   
y_coord   24.0  -18.0   -8.0    9.0  -18.0   10.0  -32.0   14.0  -35.0    8.0   

         ...  74164  74165  74166  74173  74175  74176  74177  74178  74190  \
x_coord  ...   80.0   36.0   73.0   81.0   60.0   51.0   46.0   34.0   53.0   
y_coord  ...    3.0  -29.0  -38.0    4.

In [ ]:
shots2017 = ShotMap(2017)
interactive_shot_map(shots2017)

In [ ]:
shots2018 = ShotMap(2018)
interactive_shot_map(shots2018)

In [ ]:
shots2019 = ShotMap(2019)
interactive_shot_map(shots2019)

In [ ]:
shots2020 = ShotMap(2020)
interactive_shot_map(shots2020)